# 02. Data Cleaning & Base Feature Engineering

**Stage:** 02_processing  
**Inputs:** `data/raw/{season}/fixtures.parquet`, `data/raw/{season}/match_stats.parquet`  
**Outputs:** `data/processed/clean_fixtures.parquet`  

This notebook standardizes team names across FBref and ClubElo conventions, computes home/away base match features, calculates goal/xG differentials, handles missing xG indicators without data leakage, and outputs clean fixture records.

In [1]:
# Load imports and configuration
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.config.loader import load_config
from src.processing.clean_features import clean_fixtures_pipeline

config = load_config()
season = config["data"]["seasons"][0]
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Clean season string for safe directory naming
season_clean = season.replace("/", "-")

# Construct the absolute path directly from the root
raw_dir = PROJECT_ROOT / "data" / "raw" / season_clean
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

fixtures_df = None
match_stats_df = None
clean_df = None
print(f"Processing stage initialized for season: {season}")

Processing stage initialized for season: 2024/2025


In [2]:
# Load raw datasets
fixtures_file = raw_dir / "fixtures.parquet"
stats_file = raw_dir / "match_stats.parquet"

if fixtures_file.exists():
    fixtures_df = pd.read_parquet(fixtures_file)
    print(f"Loaded raw fixtures: {len(fixtures_df)} rows")
else:
    print(f"Warning: Raw fixtures file not found at {fixtures_file}")

if stats_file.exists():
    match_stats_df = pd.read_parquet(stats_file)
    print(f"Loaded raw match stats: {len(match_stats_df)} rows")
else:
    match_stats_df = None
    print("Match stats file not found or optional")

Loaded raw fixtures: 380 rows
Loaded raw match stats: 760 rows


In [3]:
# Clean features and compute differentials
if fixtures_df is not None and not fixtures_df.empty:
    clean_df = clean_fixtures_pipeline(fixtures_df, match_stats_df)
    print(f"Cleaned features generated: {len(clean_df)} rows, {len(clean_df.columns)} columns")
else:
    print("Skipping feature cleaning: raw fixtures data not loaded")

Cleaned features generated: 380 rows, 29 columns


In [5]:
# Missing value analysis
if clean_df is not None:
    print("=== Missing Value Summary ===")
    missing_summary = clean_df.isna().sum()
    print(missing_summary[missing_summary > 0])
    print(f"is_xg_missing flag count: {clean_df["is_xg_missing"].sum()}")

=== Missing Value Summary ===
notes         380
home_xg       380
away_xg       380
home_sh       380
away_sh       380
home_sot      380
away_sot      380
xg_diff       380
shots_diff    380
sot_diff      380
dtype: int64
is_xg_missing flag count: 380


In [6]:
# Save cleaned dataset
if clean_df is not None:
    output_file = processed_dir / "clean_fixtures.parquet"
    clean_df.to_parquet(output_file, index=False)
    print(f"Saved cleaned features to {output_file}")

Saved cleaned features to /Users/mac/Documents/Projects/SportsBettingPoisson+ML/data/processed/clean_fixtures.parquet


In [7]:
# Sample display and validation
if clean_df is not None:
    print("=== Sample Cleaned Fixtures ===")
    display(clean_df[["date", "home_team", "away_team", "home_goals", "away_goals", "goals_diff", "xg_diff", "is_xg_missing"]].head())

=== Sample Cleaned Fixtures ===


,date,home_team,away_team,home_goals,away_goals,goals_diff,xg_diff,is_xg_missing
0,2024-08-16,Manchester United,Fulham,1.0,0.0,1.0,NaN,True
1,2024-08-17,Arsenal,Wolverhampton Wanderers,2.0,0.0,2.0,NaN,True
2,2024-08-17,Everton,Brighton & Hove Albion,0.0,3.0,-3.0,NaN,True
3,2024-08-17,Ipswich Town,Liverpool,0.0,2.0,-2.0,NaN,True
4,2024-08-17,Newcastle United,Southampton,1.0,0.0,1.0,NaN,True
